<a href="https://colab.research.google.com/github/zeinafarghaly-arch/ML-flyrank/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Scoring, the goal is to rank content pages by how important they are to refresh. Pages with higher opportunity_score should be reviewed first.

In [1]:
import os, sys, subprocess
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

print("Working dir:", os.getcwd())

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Pages:", len(df))
print("Correlation:", df["search_volume"].corr(df["impressions_90d"]))
print(df["trend_direction"].value_counts())
stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["opportunity_score"] = stale * visible * df["impressions_90d"]

df[["content_id", "days_since_last_update", "impressions_90d", "opportunity_score"]] \
    .sort_values("opportunity_score", ascending=False).head(10)

Working dir: /content/flyrank-ml-internship-starter
Pages: 30000
Correlation: 0.0012030310696208905
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


,content_id,days_since_last_update,impressions_90d,opportunity_score
16751,content_cf56e2e2e282,194,61678,61678
16514,content_7368877ea310,194,59472,59472
7021,content_1bfaa38ff26c,194,25715,25715
21268,content_0a91db491d14,193,13299,13299
11489,content_5feee3994adb,194,7812,7812
12045,content_c2d929d83eaa,193,7558,7558
698,content_b16bd7307b39,194,4590,4590
5327,content_fe16a55cd13d,194,4556,4556
26810,content_ecb6215e79fd,194,4429,4429
20837,content_928af3e22c80,193,1697,1697


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

There is no column that directly says a page should be refreshed, so I use trend_direction == "down" as a proxy to represent pages that are currently declining.

In [2]:
df["is_declining"] = (df["trend_direction"] == "down").astype(int)
print("Declining rate:", round(df["is_declining"].mean(), 3))
df["trend_direction"].value_counts()


Declining rate: 0.542


,count
trend_direction,
down,16262
stable,5962
up,4388
new,2236
flat,1152


## 3. Success metric

*One metric you can defend. What number means 'good'?*

I use Precision@500. I rank pages by opportunity_score and check how many of the top 500 are actually declining.

In [3]:
import numpy as np

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

k = 500
print(f"Precision@{k}: {precision_at_k(df['opportunity_score'], df['is_declining'], k):.3f}")

Precision@500: 0.550


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Each row represents one content page (content_id) from one of 32 anonymized clients. It includes the page ID, the data used to calculate opportunity_score, and the decline status used to check how well the score works.

In [4]:
print("Rows:", len(df), "| Unique clients:", df["client_id"].nunique())
df[["content_id", "client_id", "days_since_last_update", "impressions_90d",
    "trend_direction", "is_declining", "opportunity_score"]].head()

Rows: 30000 | Unique clients: 32


,content_id,client_id,days_since_last_update,impressions_90d,trend_direction,is_declining,opportunity_score
0,content_304f48230142,client_f369cb89fc,20,3803,down,1,0
1,content_a1fb4e703a9e,client_4e07408562,25,15320,down,1,0
2,content_9aa793d4d895,client_7f2253d7e2,20,12581,down,1,0
3,content_331d6c4de07b,client_19581e27de,22,11751,stable,0,0
4,content_d99b7a2d90ca,client_3fdba35f04,14,19140,down,1,0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A simple rule is not enough because:

No single feature clearly identifies declining pages.
One feature alone does not perform well.
Missing values vary by content type, making fixed rules difficult.

An ML model can combine multiple features and learn these patterns automatically.

In [5]:
for col in ["days_since_last_update", "impressions_90d", "avg_position", "ctr", "engagement_rate"]:
    print(col, "correlation with is_declining:", round(df[col].corr(df["is_declining"]), 3))

days_since_last_update correlation with is_declining: 0.081
impressions_90d correlation with is_declining: -0.018
avg_position correlation with is_declining: -0.029
ctr correlation with is_declining: -0.062
engagement_rate correlation with is_declining: -0.013


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.